# **Perineuronal net morphology (PNN) morphology analysis notebook**

<ins>**Author:**</ins> Shannon Rhoads (Github @shanrhoads)

<ins>**Notebook version:**</ins> v1.2

<ins>**Purpose:**</ins> The purpose of this notebook is to facilitate a pipeline for quantitative analysis of perineuronal net (PNN) morphology from high resolution fluorescence microscopy images. The images this pipeline was optimized for were 3D (XYZ) STED micrographs taken with a Leica STELLARIS 8 FALLCON STED microscope. Neurons from mouse tissue were stained with WFA to detect the PNN component N-acetylgalactosamine.

## **Getting started**
#### *Only do this setup process once per computer or location you will run the analysis.*
1. Clone this repository to the local or remote computer you will use to run the analysis:
    - In your computers terminal, naviate to the location you wish to store the clone of the repository
    - Execute the following code in your computer's terminal:
        > ``` Python
        > git clone https://github.com/shanrhoads/PNN-morpho-quant.git
        > ```
2. Follow the instruction in the [env_create.sh](/env_create.sh) file to install the necessary packages into a new Python environment. 
2. Download [Visual Studio Code](https://code.visualstudio.com/Download), or use Jupyter labs (already install in the conda environemnt) to access the files in this repository.
3. Open this file in VSCode or Jupyter Labs.
    - For VSCode, open the program on your computer and use the built in firstory naviation tools to find and open this file.
    - For Jupyter Labs access, open a terminal on your computer, activate your conda environment, and run "jupyter labs" command to initiate the Jupyter labs interface. From there you can navigate to this file using the built in directory navigation tools.
4. Click on `Select Kernel` in the top right; choose `Python Environment...` and either `PNN-morpho` or `base` from the list of options based on your choice above.

## **Notebook organization:**

This notebook includes the following sections that should be run in order for each dataset:
1. **Imports** - this section imports the necessary Python packages and functions to run the analysis pipeline below (required each run).
2. **Testing Analysis Settings (on select single images)** - This section explains the individual steps included in the final segmentation, skeletonization, and quantification batch process functions. When beginning the analysis for a new, independent dataset, utilize this section to optimize the segmentation and skeletonization parameters before batch processing. It is recommended to test the selected settings on a few images across biological replicates (if possible) and experimental conditions to increase the chances of choosing settings that will be broadly applicable for your data. You're chosen settings will never work perfectly one very image, so you are aiming to find a balance of over and undersegmentation, for example, that will work for most images. After applying the chosen settings to an entire dataset or replicate of data, step 4 below then help you refine any images that the chosen settings did not work for.
3. **Batch Process segmentation and skeletonization (all images in one data folder)** - This section allows you to apply your chosen settings to a set of images from a single folder. The segmentation and skeletonization output files will be saved in a separate specified location. This section is intended to be run separately for data from each biological replicate (contained in one folder). You can run multiple batches sequentially if you'd like to process more than one folder of data.
4. **Quality Check segmentations** - This step is necessary to ensure all of the images have a segmentation that accurately reflects the PNN instensity image. Again, the expectation is not that each image will be perfect, but if any images are very far off from accurate, they can easy skew your data. You will use the code blocks included in Step 2 and editting tools in napari to edit any files that had insufficient segmentations.
5. **Batch Process morphological quantification (all images in one data folder)** - Once the segmentations (and skeletons) have been batch processed and visually inspected for accuracy, the data from one biological replicate can be quantified in this step. The inputs include the raw intensity image used for segmentation/skeletonization and the segmentation/skeletonization files. You can run multiple batches sequentially if you'd like to process more than one folder of data.
6. **Summarize quantitative data per image (quantitative data from multiple folders)** - All of the quantitative data from multiple biological replicates is then summarized per cell. The input is intended to include a list of file paths to all of the quantitative data that will be included during statistical analysis (all biological replicates), though is can also be applied to a single folder of data if only one is listed in the input.

## **Recommended data organization:**

It is recommended to maintain the following file structure:

``` bash
experiment-name-1/                                                                      # included as "experiment" metdata
├── Male/                                                                               # included as "sex" metadata
    ├── Pair#/                                                                          # included as "replicate" metadata
    |   ├── WT/                                                                         # input for steps 3 & 5; included as "genotype" metadata
    |   |   └── cell-num_region_subject-ID_decon_ch02.tif                               # raw input file
    |   ├── cKO/                                                                        # input for steps 3 & 5; included as "genotype" metadata
    |   |   └── cell-num_region_subject-ID_decon_ch02.tif                               # raw input file
    |   ├── processing-data_WT-seg-skel/                                                # result of step 3; input for step 5
    |   |   ├── cell-num_region_subject-ID_decon_ch02-PNN_instance_seg.tif              # instance segmentation of the PNN
    |   |   └── cell-num_region_subject-ID_decon_ch02-PNN_skeleton.tif                  # skeleton of the instance segmentation
    |   ├── processing-date_cKO-seg-skel/                                               # result of step 3; input for step 5
    |   |   ├── cell-num_region_subject-ID_decon_ch02-PNN_instance_seg.tif              # instance segmentation of the PNN
    |   |   └── cell-num_region_subject-ID_decon_ch02-PNN_skeleton.tif                  # skeleton of the instance segmentation
    |   ├── processing-date_WT-quant/                                                   # result of step 5; input for step 6
    |   |   └── cell-num_region_subject-ID_decon_ch02-PNN_quantification.csv            # output quantification (one row of data per PNN piece quantified)
    |   └── processing-date_cKO-quant/                                                  # result of step 5; input for step 6
    |       └── cell-num_region_subject-ID_decon_ch02-PNN_quantification.csv            # output quantification (one row of data per PNN piece quantified)
    └── Pair#/                                                                          # included as "replicate" metadata
|   |   └── ...
├── Female/
|   └── ...
├── Summary-quantification/                                                         # result of step 6
    └── summary-stats.csv                                                           # per image summary statistics (one row of data per image)
experiment-name-2/
└── ...
```

__________
## **STEP 1. Imports:**
Below, the packages/functions necessary for this analysis are imported. This should be run everytime any part of this notebook is to be run.

In [ ]:
# IMPORTS
from pathlib import Path
from typing import Union, List #,Tuple, Any
import time

from bioio import BioImage
# from bioio.writers import OmeTiffWriter
# from tifffile import imwrite

import napari

# from dask_image.imread import imread
import numpy as np
import skimage
import skan
import matplotlib.pyplot as plt
from scipy import stats
# import infer_subc

import sys
sys.path.insert(0, str(Path("..").resolve()))
from src.image_processing import skeletonize_plus, batch_PNN_seg_skel
from src.quantification import surface_area_from_props, batch_PNN_quant, batch_summarize_quant

import pandas as pd
pd.set_option('display.max_columns', None)

----------

## **STEP 2. Testing Analysis Settings (on select single images):**

Below, you will find the individual steps that are used to process the image segmentation, skeletonization, and quantification. This section only needs to be run if you are optimizing settings to use in batch processing segmentation/skeletonization or modifying particular files after batching is finished.

### **1. Read file and metadata**

#### **1A. List files in path**

Specify the following information:
- `file_path`: file path where the input images are located written as a string
- `file_type`: input file type as a string (e.g., ".tif")

Then run the cell below to read in the list of files of the specified type from the specified location. A Napari window will also pop up. The outputs of each processing step below will be added to the window as new layers.

In [5]:
### USER INPUTS ###
file_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair 5/3D STED/cKO"  # OPTIONS:  WT" cKO"
file_type = ".tif"



### PROCESSING - no edits below ###
# open a Napari viewer window to visualize images & processing steps
viewer = napari.Viewer()

# create sorted list of files in directory with specified file type
file_list = sorted(Path(file_path).glob(f"*{file_type}"))

# print list of files with associated index number for selection below
pd.set_option('display.max_colwidth', None)
pd.DataFrame({"Image Name":file_list})

,Image Name
0,/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair 5/3D STED/cKO/3_VCX_382_5_decon_ch02.tif
1,/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair 5/3D STED/cKO/4_VCX_382_5_decon_ch02.tif
2,/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair 5/3D STED/cKO/5_VCX_382_5_decon_ch02.tif
3,/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair 5/3D STED/cKO/6_VCX_382_5_decon_ch02.tif
4,/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair 5/3D STED/cKO/7_VCX_382_5_decon_ch02.tif


#### **1B. Select file of interest for testing**

Specify the following information:
- `file_index`: the index of the image you would like to look at in the analysis below. The index value is found to the left of the file paths listed in the table above.

Then run the cell below to read  the image
 and view some of its metadata.

In [7]:
### USER INPUTS ###
file_index = 2  # change this number to select a different file from the list above



### PROCESSING - no edits below ###
# read in file
raw_file = BioImage(str(file_list[file_index]))
raw_img = raw_file.data
metadata = raw_file.standard_metadata

# save relevant metadata information as objects for use later
voxel_size_ZYX = (raw_file.physical_pixel_sizes.Z, raw_file.physical_pixel_sizes.Y, raw_file.physical_pixel_sizes.X)

# print relevant info about the image
print("File shape:", raw_file.shape)
print("Dimensions:", raw_file.dims)
print("Image channels:", raw_file.channel_names)
print("Voxel size:", raw_file.physical_pixel_sizes)



### ALTERNATIVE - if .lif file/metadata desired ###
# # read in .lif file
# lif_path = r"W:\Baldwin Lab\Hayli Spence-Osorio\PNN Project Data\Pair 4\Metadata\250429_HSO_PNN_Pair_4_Day_1.lif"
# lif_imgs = BioImage(lif_path)

# # extract raw image data and metadata
# lif_img_raw = lif_imgs.data
# lif_metadata = lif_imgs.metadata

# # view image in napari
# viewer.add_image(lif_img_raw, scale=voxel_size_ZYX, name="Deconvolved PNN image")

#####################################################################
### DEPRICATED ###
# # alternative read approach using dask array
# # the dask array approach before is formatted to utilize a series of tif images that separate channels, z's, time, etc.. 
# # Zarr formatting may still be the best approach for single files
# stack = imread(str(file_list[file_index]))
# napari.imshow(stack, multiscale=False)

Attempted file (/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair 5/3D STED/cKO/5_VCX_382_5_decon_ch02.tif) load with reader: <class 'bioio_ome_tiff.reader.Reader'> failed with error: bioio-ome-tiff does not support the image: '/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair 5/3D STED/cKO/5_VCX_382_5_decon_ch02.tif'. Failed to parse XML for the provided file. Error: not well-formed (invalid token): line 1, column 6


File shape: (1, 1, 72, 1736, 1736)
Dimensions: <Dimensions [T: 1, C: 1, Z: 72, Y: 1736, X: 1736]>
Image channels: [np.str_('Channel:0:0')]
Voxel size: PhysicalPixelSizes(Z=0.072295, Y=0.01675099974733943, X=0.01675099974733943)


#### **4C. Select small subregion in image to speedup processing below**

Specify the following information:
- `use_small_region`: True/False to indicate if you would like to only process a smaller portion of the image (memory saving step) in the steps below, or not. *Note: if you would like to change which chunk of the image you are selecting, you can adjust the coordinates being selected within the square brackets following np.squeeze()[HERE].* 

Then run the cell below to select the specified small portion of your image, or the entire image. This is the image you will use to optimize the settings for each step below. The output can be visualized in the Napari window.

In [8]:
### USER INPUTS ###
use_small_region = True  # set to True to use a small region of the image for testing purposes 


### PROCESSING - no edits below ###
# select small region or full image for testing analysis
if use_small_region:
    test_img = np.squeeze(raw_img)[30:45, 800:1250, 600:1050]
else:
    test_img = np.squeeze(raw_img)

# visualize image in napari
viewer.layers.clear()
viewer.add_image(test_img, scale=voxel_size_ZYX, name="Deconvolved PNN image")
print("The test image has been added to the Napari viewer.")

The test image has been added to the Napari viewer.


### **2. Segment PNN**

#### **2A. Rescale intensity values**


No user input is required.

Run the cell below to rescale the intensity values in the image. This produces a new image array where the max value is 1 and the minimum value is 0. This should help normalize the segmentation outcomes across images if the value ranges vary by image.

rescaled_image = (original_image_values - minimum_value) / (maximum_value - minimum_value)

*Note: a very small value (1x10^-8) is added to the minimum value to ensure we will not be dividing by 0.*

In [10]:
### PROCESSING - no edits below ###
# calculate min/max values
strech_min = test_img.min()
strech_max = test_img.max()

# rescale image
rescale = (test_img - strech_min + 1e-8) / (strech_max - strech_min + 1e-8)

# visualize outputs
viewer.add_image(rescale, scale=voxel_size_ZYX, name=f"Rescaled")

<Image layer 'Rescaled' at 0x7f32a29c5820>

#### <mark> ** THESE STEPS NOT USED - THEY TAKE A TON OF TIME TO PROCESS** </mark>
#### ~~**2A. Background subtraction**~~ 

~~[`Rolling ball background subtraction`](https://scikit-image.org/docs/0.25.x/auto_examples/segmentation/plot_rolling_ball.html) can be used to remove non-uniform background from images before segmentaiton or intensity measurements. In this process the amount of background per pixel/voxel is calculated from a region about the image (here defined as the radius). The radius (in voxels) can be adjusted to modify the effects of the rolling ball algorithm; it should be larger than the largest object/structure of interest in your image.~~



In [ ]:
# ### USER INPUT ###
# bg_radius = 50

# # calculate background per pixel using the rolling ball method
# bg = skimage.restoration.rolling_ball(test_img, radius=bg_radius)
# bg_subtract = test_img - bg

# # visualize output
# viewer.add_image(bg_subtract, scale=voxel_size_ZYX, name="Background subtracted")

#### ~~**2A. Denoising**~~ <mark> 

~~There is quite a bit of speckley noise in your image that is making segmentation of the PNN intensity more difficult. Below, [`skimage.restoration`](https://scikit-image.org/docs/0.25.x/api/skimage.restoration.html#skimage.restoration.denoise_bilateral) module is used to denoise the image.~~

In [ ]:
# denoised = skimage.restoration.denoise_nl_means(test_img, patch_size=50, patch_distance=100)

# viewer.add_image(denoised, scale=voxel_size_ZYX, name="Denoised")

#### **2B. Smoothing**

[`Gaussian`](https://scikit-image.org/docs/0.25.x/api/skimage.filters.html#skimage.filters.gaussian) and [`median`](https://scikit-image.org/docs/0.25.x/api/skimage.filters.rank.html#skimage.filters.rank.median) smoothing filters are commonly used to smooth images and reduce certain types of noise, like the high salt and pepper noise commonly present in your WFA stained images. Each filters smooth the image different ways and are commonly used in combination. I've included both options here with sigma/size filter values that can be used to adjust how much smoothing occurs (large values = more smoothing).

Specify the following information:
- `gaus_sigma`: the sigma value used for gaussian smoothing. The higher the number, the more the image is smoothed
- `med_size`: the window/neighborhood size used for the median smoothing filter. The higher the number, the more the image is smoothed

Then run the cell below to apply the smoothing filters. The result can be visualized in the Napari window. *Note: this step is hard coded to work with 3D image (XYZ); 2D images will cause an error, most likely.*

In [11]:
### USER INPUT ###
gaus_sigma = 2
med_size = 8



### PROCESSING - no edits below ###
# applying smoothing filters
if gaus_sigma:
    smoothed = skimage.filters.gaussian(test_img, sigma=gaus_sigma)
else:
    smoothed = test_img

if med_size:
    fp = skimage.morphology.footprint_rectangle((round(med_size*(voxel_size_ZYX[0]/float(np.max(voxel_size_ZYX)))), 
                                                 round(med_size*(voxel_size_ZYX[1]/float(np.max(voxel_size_ZYX)))), 
                                                 round(med_size*(voxel_size_ZYX[2]/float(np.max(voxel_size_ZYX))))))
    smoothed = skimage.filters.median(smoothed, footprint=fp)

# visualize outputs
viewer.add_image(smoothed, scale=voxel_size_ZYX, name=f"Smoothed: gaus={gaus_sigma}, med={med_size}")

<Image layer 'Smoothed: gaus=2, med=8' at 0x7f32a0e3f500>

#### **2C. Thresholding** (multiple options below; choose 1)

There any many types of thresholding methods available in Python. The simplest form is to apply a manual threshold where you determine a cutoff value to apply to the entire image (all pixels/voxels with this intensity value and above will be included in the segmentation). Alternatively, the [`skimage`](https://scikit-image.org/docs/stable/api/skimage.filters.html) package has many mathematical approaches to calculate an "appropriate" threshold value based on the intensity values in the image. These automated thresholds can help to adjust segmentation outcomes based on image-to-image variations. 

Both manual and automated thresholding approaches can be applied to the entire image (globally). Automated thresholds can also be applied in a "local" or "adaptive" fashion, where a small local region surround each voxel is used to set a specific threshold value per voxel. If the intensity value is equal to or greater than the calculated threshold value, that voxel is included in the segmentation. Local thresholding can help optimze the segmentation outcomes for images with varying intensities across different regions in the image (e.g., higher background on one side or area of the image than the rest).

Test each of the approaches below and **choose one** to continue with in the subsequent steps (you will be prompted to choose which is the best below).

<ins>**Approach 1:**</ins> Manual thresholding

In this approach, a cutoff value, representing the minimum intensity value to include as part of the segmentation, is specified by the user. Any voxels with an intensity value less than the cutoff will be excluded from the semantic segmentation.

Specify the following:
- `manual_cutoff`: The minimum intensity value you wish to include in your segmentation. You can use the smoohted image layer in the napari viewer to explore the intensity values inside your objects of interest.

Then run the cell below to apply the cutoff value to your image.

In [12]:
### USER INPUT ###
manual_cutoff = 0.045



### PROCESSING - no edits below ###
# select everything above threshold value for segmentation
seg = smoothed >= manual_cutoff

# visualize segmentation
viewer.add_image(seg, scale=voxel_size_ZYX, name=f"Segmentation: thresh={manual_cutoff}", blending="additive", opacity=0.4, colormap='magenta')

<Image layer 'Segmentation: thresh=0.045' at 0x7f32a07a6a50>

<ins>**Approach 2:**</ins> Automated thresholding

In this approach, the threshold cutoff value per image is calculated based on the range of intensity values within that image. The automated thresholding approaches included in the ['skimage'](https://scikit-image.org/docs/stable/api/skimage.filters.html) package can be applied here.

Specify the following:
- `automated_method`: the name of the automated thresholding method. Options include: 'otsu', 'multiotsu', 'li', 'yen', 'isodata', 'triangle', 'minimum', 'mean'. You can read more about each method here: ['skimage.filters` documentation](https://scikit-image.org/docs/0.23.x/api/skimage.filters.html#module-skimage.filters) and [thresholding examples](https://scikit-image.org/docs/0.23.x/auto_examples/segmentation/plot_thresholding.html).
- `adjust`: this value is used to adjust the threshold value calculated from the automated threshold. A value of 1 applies no adjustment, a value <1 will make the threshold more lenient (more stuff included), and a value >1 will make the threshold more stringent (less stuff included).
- `multiotsu_middle_to`: *only necessary is "multiotsu" was chosen as the automated_method* Multiotsu threshold is set to divide the image intensities into three classes (lowest, middle, highest). This setting lets you choose if the middle group should be included in the segmentation ('foreground') or as part of the background ('background').

Then run the cell below to apply your chosen thresholding approach to the entire image. The resulting semgnetation is output in the Napari window.

In [13]:
### USER INPUT ###
threshold_method = 'multiotsu'      # OPTIONS: 'otsu', 'li', 'yen', 'isodata', 'triangle', 'minimum', 'mean', 'multiotsu'
adjust = 0.55                      # OPTIONS: 1 = no threshold adjustment, <1 = more stuff selected, >1 = less stuff selected
multiotsu_middle_to = 'background'  # OPTIONS: 'foreground', 'background'




### PROCESSING - no edits below ###
# Apply automated threshold based on selected method
if threshold_method == 'otsu':
    thresh_value = skimage.filters.threshold_otsu(smoothed)
elif threshold_method == 'multiotsu':
    thresholds = skimage.filters.threshold_multiotsu(smoothed, classes=3)
    if multiotsu_middle_to == 'foreground':
        thresh_value = thresholds[0]  # select the second highest threshold
    elif multiotsu_middle_to == 'background':
        thresh_value = thresholds[1]   # select the lowest threshold
    else:
        raise ValueError(f"Unrecognized multiotsu middle to option: {multiotsu_middle_to}")
elif threshold_method == 'li':
    thresh_value = skimage.filters.threshold_li(smoothed)
elif threshold_method == 'yen':
    thresh_value = skimage.filters.threshold_yen(smoothed)
elif threshold_method == 'isodata':
    thresh_value = skimage.filters.threshold_isodata(smoothed)
elif threshold_method == 'triangle':
    thresh_value = skimage.filters.threshold_triangle(smoothed)
elif threshold_method == 'minimum':
    thresh_value = skimage.filters.threshold_minimum(smoothed)
elif threshold_method == 'mean':
    thresh_value = skimage.filters.threshold_mean(smoothed)
else:
    raise ValueError(f"Unrecognized threshold method: {threshold_method}")

# Apply threshold with optional adjustment
seg_auto = smoothed >= thresh_value*adjust

# Print the calculated threshold value for reference
print(f"Calculated threshold value using {threshold_method}: {thresh_value*adjust}")

# Visualize segmentation
viewer.add_image(seg_auto, scale=voxel_size_ZYX, name=f"Auto seg: {threshold_method} (thresh={thresh_value*adjust})", blending="additive", opacity=0.4, colormap='green')


# ### FOR TESTING ALL THE METHODS ###
# ### Compare multiple automated threshold methods ###
# # the following script can be used to visualize the different thresholding methods available in skimage
# methods = ['otsu', 'li', 'yen', 'isodata', 'triangle', 'minimum', 'mean', 'multiotsu']
# multiotsu_middle = 'foreground'  # OPTIONS: 'foreground', 'background'

# for method in methods:
#     if method == 'otsu':
#             thresh_val = skimage.filters.threshold_otsu(smoothed)
#     elif method == 'multiotsu':
#         thresholds = skimage.filters.threshold_multiotsu(smoothed, classes=3)
#         if multiotsu_middle == 'foreground':
#             thresh_val = thresholds[0]  # select the second highest threshold
#         elif multiotsu_middle == 'background':
#             thresh_val = thresholds[1]   # select the lowest threshold
#         else:
#             raise ValueError(f"Unrecognized multiotsu middle to option: {multiotsu_middle}")
#     elif method == 'li':
#         thresh_val = skimage.filters.threshold_li(smoothed)
#     elif method == 'yen':
#         thresh_val = skimage.filters.threshold_yen(smoothed)
#     elif method == 'isodata':
#         thresh_val = skimage.filters.threshold_isodata(smoothed)
#     elif method == 'triangle':
#         thresh_val = skimage.filters.threshold_triangle(smoothed)
#     elif method == 'minimum':
#         thresh_val = skimage.filters.threshold_minimum(smoothed)
#     elif method == 'mean':
#         thresh_val = skimage.filters.threshold_mean(smoothed)
#     else:
#         raise ValueError(f"Unrecognized threshold method: {method}")
    
#     seg_temp = smoothed >= thresh_val*adjust
#     print(f"{method}: threshold = {thresh_val*adjust}")
#     viewer.add_image(seg_temp, scale=voxel_size_ZYX, name=f"{method} ({thresh_val*adjust})", blending="additive", opacity=0.4, colormap='green')

Calculated threshold value using multiotsu: 0.026929464504657365


<Image layer 'Auto seg: multiotsu (thresh=0.026929464504657365)' at 0x7f32a2b18560>

<ins>**Approach 3:**</ins> Local, automated thresholding 

Above, the threshold was applied globally (to the whole image). Below, we will apply the threshold locally. This determines the threshold value for each pixel/voxel in the image based on the intensity values in a local region around it. The local region size can be modified as needed to adjust the thresholding outcomes. The approach could improve segmentation in areas where the intensity range or amount of background varies across different regions within the same image.

Specify the following information:
- `local_method`: the name of the local thresholding method. Options include: 'otsu', 'mean', 'gaussian'. You can read more about each method here: ['skimage.filters` documentation](https://scikit-image.org/docs/0.23.x/api/skimage.filters.html#module-skimage.filters) and [thresholding examples](https://scikit-image.org/docs/0.23.x/auto_examples/segmentation/plot_thresholding.html)
- `local_size`: the size of the local area around each voxel to consider when calculating the threshold. The value must be an odd integer. Larger numbers include larger areas; this number should likely be larger than the width of your object.
- `gaussian_sigma_local`: if 'gaussian' was chosen as the local_method, specify the size of the gaussian sigma

*Note: <ins>this approach is MUCH SLOWER</ins> for each image as it has to calculate the threshold cutoff for each voxel separately.*

In [15]:
### USER INPUT ###
# determine the method to use for local thresholding
local_method = 'otsu'       # OPTIONS: 'gaussian', 'mean', 'otsu'
local_size = 101            # must be odd; large sizes for larger structures, small sizes for smaller structures
# 71 = 2 min 57 s
# Some methodsrelevant parameters
guassian_sigma_local = 40    # only used if local_method is 'gaussian'

# rescale and convert to 8-bit for local thresholding
max_val = smoothed.max()
smoothed_8bit = np.round((smoothed / max_val)*255).astype(np.uint8)
smoothed_8bit_downscaled = (smoothed_8bit // 2).astype(np.uint8)

### PROCESSING - no edits below ###
# Calculate global automated threshold value
if local_method == 'otsu':
    # create footprint for local region
    footprint = skimage.morphology.ball(local_size)
    local_otsu_threshold = skimage.filters.rank.otsu(smoothed_8bit_downscaled, footprint)
    seg_local_auto = smoothed_8bit_downscaled > local_otsu_threshold
elif local_method == 'mean':
    local_mean_threshold = skimage.filters.threshold_local(smoothed_8bit, block_size=local_size, method='mean')
    seg_local_auto = smoothed_8bit >= local_mean_threshold
### MEDIAN LOCAL THRESHOLD NOT WORKING ###
# elif local_method == 'median':
#     local_median_threshold = skimage.filters.threshold_local(smoothed, block_size=local_size, offset=0.1, method='median')
#     seg_local_auto = smoothed >= local_median_threshold
elif local_method == 'gaussian':
    local_gaussian_threshold = skimage.filters.threshold_local(smoothed_8bit, block_size=local_size, method='gaussian', param=guassian_sigma_local)
    seg_local_auto = smoothed_8bit >= local_gaussian_threshold
### NOT TESTED YET ###
# elif local_method in ['li', 'yen']:
#     if local_method == 'li':
#         def thresh_method(neighborhood):
#             funct = skimage.filters.threshold_li(neighborhood)
#             return funct
#     elif local_method == 'yen':
#         def thresh_method(neighborhood):
#             funct = skimage.filters.threshold_yen(neighborhood)
#             return funct
#     elif local_method == 'isodata':
#         def thresh_method(neighborhood):
#             funct = skimage.filters.threshold_isodata(neighborhood)
#             return funct
#     elif local_method == 'triangle':
#         def thresh_method(neighborhood):
#             funct = skimage.filters.threshold_triangle(neighborhood)
#             return funct
#     elif local_method == 'minimum':
#         def thresh_method(neighborhood):
#             funct = skimage.filters.threshold_minimum(neighborhood)
#             return funct
    
#     local_thresholds = skimage.filters.threshold_local(smoothed, block_size=local_size, method='generic', param=thresh_method)
#     seg_local_auto = smoothed >= local_thresholds
else:
    raise ValueError(f"Unrecognized local threshold method: {local_method}")

viewer.add_image(seg_local_auto, scale=voxel_size_ZYX, name=f"Local {local_method.capitalize()} segmentation", blending="additive", opacity=0.4, colormap='cyan')



### FOR REFERENCE - this works to process local otsu ###
# local_size = 100      ## For [15,450,450] sized image: size=100 -> ~9mins (pretty good outcome); size=11 much faster (created too many smaller objs); size=50 --> 2 mins, 3 sec (similar outcome as manual seg)

# smoothed_uint8 = (smoothed / smoothed.max() * 255).astype(np.uint8)

# footprint = skimage.morphology.ball(local_size)

# local_otsu_threshold = skimage.filters.rank.otsu(smoothed_uint8, footprint)
# binary_local_otsu = smoothed_uint8 > local_otsu_threshold

# viewer.add_image(smoothed_uint8, scale=voxel_size_ZYX, name=f"uint8 smoothed image")
# viewer.add_image(local_otsu_threshold, scale=voxel_size_ZYX, name=f"Local Otsu threshold (radius={local_size})")
# viewer.add_image(binary_local_otsu, scale=voxel_size_ZYX, name=f"Local Otsu segmentation (radius={local_size})", blending="additive", opacity=0.4, colormap='cyan')

<Image layer 'Local Otsu segmentation [1]' at 0x7f32a28d8680>

#### **2D. Clean-up segmentation**

Segmentations can have errors due to imperfections in the thresholding output. Two different refining setups have been added below: removing small objects and filling small holes.

#### Remove small objects:

In [ ]:
### USER INPUTS ###
obj_min_diameter = 10
obj_method = '3D' # OPTIONS: 'slices' or '3D'
seg = seg_local_auto  # choose which segmentation to use for object filtering; OPTIONS: seg, seg_auto, seg_local_auto


### PROCESSING - no edits below ###
# filter objects based on size
if obj_method == 'slices':
    filtered = np.zeros_like(seg)
    for z in range(seg.shape[0]):
        input = seg[z,:,:]
        seg_size_filter = skimage.morphology.remove_small_objects(input, min_size=obj_min_diameter**2)
        input = np.expand_dims(seg_size_filter, axis=0)
        filtered[z,:,:] = input
elif obj_method == '3D':
    filtered = skimage.morphology.remove_small_objects(seg, min_size=obj_min_diameter**3)
else:
    SyntaxError("Unrecognized method chosen. Options include: 'slices' or '3D'.")

# visualize
viewer.add_image(filtered, scale=voxel_size_ZYX, name=f"Filter obj: method={obj_method}, obj={obj_min_diameter}", blending="additive", opacity=0.4, colormap='cyan')

#### Fill small holes

In [ ]:
### USER INPUTS ###
small_hole_diameter_max = 0
hole_method = 'slices' # OPTIONS: 'slices' or '3D'


### PROCESSING - no edits below ###
# fill holes based on size
if hole_method == 'slices':
    filled = np.zeros_like(filtered)
    for z in range(filtered.shape[0]):
        input = filtered[z,:,:]
        seg_fill_holes = skimage.morphology.remove_small_holes(input, small_hole_diameter_max**2, connectivity=8)
        input = np.expand_dims(seg_fill_holes, axis=0)
        filled[z,:,:] = input
elif hole_method == '3D':
    filled = skimage.morphology.remove_small_holes(filtered, small_hole_diameter_max**3, connectivity=26)
else:
    SyntaxError("Unrecognized method chosen. Options include: 'slices' or '3D'.")

# visualize
viewer.add_image(filled, scale=voxel_size_ZYX, name=f"Fill holes: method={hole_method}, hole={small_hole_diameter_max}", blending="additive", opacity=0.4, colormap='cyan')

#### **2E. Create instance segmentation**

The instance segmentation will not be used for skeletonization, but will be able to tell us how many separate pieces of the PNN exist.

In [ ]:
### PROCESSING - no edits below ###
# create instance seg
instance_seg = skimage.morphology.label(filled)

# visualize output
viewer.add_labels(instance_seg, scale=voxel_size_ZYX, name=f"Instance segmentation", opacity=0.4)

##### ***Optional:* save the skeleton image for reference later**

In [ ]:
### USER INPUTS ###
seg_output_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair_5/3D_STED/WT_seg-skel_20260427-single-out" 

### PROCESSING - no edits below ###
# save instance segmentation to output path
skimage.io.imsave(f"{seg_output_path}/{str(file_list[file_index].stem)}-PNN_instance_seg.tif", instance_seg, plugin="tifffile")

### **3. Skeletonize segmentation**

[Skeletonization](https://scikit-image.org/docs/0.25.x/auto_examples/edges/plot_skeleton.html) is the process by which a 2D or 3D object is narrowed to a pixel-wide representation of the original area/volume. Then, the skeleton is converted into a network graph using the [`skan`](https://skeleton-analysis.org/stable/) package for easier downstream manipulations.

The napari visualization in the next step shows the skeleton branches color coded by length.

#### **3A. Create skeleton & summarize information about each individual branch**

The summary table includes the following information:
- `skeleton_id`: Unique ID for each separate skeleton object (derived from the instance segmentation label ID)
- `branch_id`: Sequential unique ID for each branch/path in the skeleton (0 to n_paths-1)
- `random_branch_id`: Randomly permuted branch ID for visualization purposes
- `node_id_src`: Pixel index ID of the source/starting node of the branch
- `node_id_dst`: Pixel index ID of the destination/ending node of the branch
- `branch_distance`: Total distance along the branch path in physical units (µm), accounting for pixel spacing
- `branch_type`: Classification of the branch topology:
    - 0 = endpoint-to-endpoint (isolated branch)
    - 1 = junction-to-endpoint
    - 2 = junction-to-junction
    - 3 = isolated cycle
- `mean_pixel_value`: Mean intensity value of pixels along the branch path
- `stdev_pixel_value`: Standard deviation of intensity values along the branch path
- `image_coord_src_0`, `image_coord_src_1`, `image_coord_src_2`: Source node coordinates in image space (pixels) for Z, Y, X respectively
- `image_coord_dst_0`, `image_coord_dst_1`, `image_coord_dst_2`: Destination node coordinates in image space (pixels) for Z, Y, X respectively
- `coord_src_0`, `coord_src_1`, `coord_src_2`: Source node coordinates in physical space (µm) for Z, Y, X respectively
- `coord_dst_0`, `coord_dst_1`, `coord_dst_2`: Destination node coordinates in physical space (µm) for Z, Y, X respectively
- `euclidean_distance`: Straight-line (Euclidean) distance between source and destination nodes in physical units (µm)

In [ ]:
### PROCESSING - no edits below ###
# create skeleton from the instance segmentation
labeled_skel, skeleton = skeletonize_plus(instance_seg)

# convert to network graph based  labeled skeleton object
skel_g = skan.Skeleton(labeled_skel, spacing=voxel_size_ZYX, value_is_height=False)

# visualize output
viewer.layers.clear()
viewer.add_image(smoothed, scale=voxel_size_ZYX, name=f"Smoothed Input")
viewer.add_image(filled, scale=voxel_size_ZYX, name=f"Segmentation", blending="additive", opacity=0.3)
viewer.add_labels(instance_seg, scale=voxel_size_ZYX, name=f"Instance segmentation", blending="additive", opacity=0.8)

all_paths = [skel_g.path_coordinates(i) for i in range(skel_g.n_paths)]
paths_table = skan.summarize(skel_g, separator='_')
paths_table.insert(1, 'branch_id', np.arange(skel_g.n_paths))
paths_table.insert(2, 'random_branch_id', np.random.default_rng().permutation(skel_g.n_paths))

# replace skeleton_id with the ID of the organelle object from which the skeleton branch originated
if not np.any(skel_g.path_stdev()):
    # checker to see if all path points and nodes come from the same object
    paths_table['skeleton_id'] = skel_g.path_means().astype(int)
else:
    raise ValueError("at least one branch came from different organelle objects")

# calculate the degree of connectivity for each branch end point
endpoints_src = skel_g.paths.indices[skel_g.paths.indptr[:-1]]
endpoints_dst = skel_g.paths.indices[skel_g.paths.indptr[1:] - 1]

deg_src = skel_g.degrees[endpoints_src]
deg_dst = skel_g.degrees[endpoints_dst]
paths_table['deg_src'] = deg_src
paths_table['deg_dst'] = deg_dst

# view skeleton and paths table
viewer.add_shapes(all_paths, shape_type='path', properties=paths_table, edge_width=1, edge_color='skeleton_id', edge_colormap='hsv', scale=voxel_size_ZYX, opacity=1, name="Skeleton")
paths_table.set_index(['skeleton_id', 'branch_id']).sort_index()

#### **3B. Refine skeleton**
Based on measures calculated by the `skan` package, the skeleton can be refined.

Here, I've chosen to refine the skeleton by removing the shortest branches, but any of the skeleton metrics included in the table below could be used to refine the skeleton.

In [ ]:
### PROCESSING - no edits below ###
# plot the histogram of branch lengths using plt.hist from matplotlib package
b_len = paths_table['branch_distance']
possible_range = (b_len.min(), b_len.max())
num_bins = round(possible_range[1]-possible_range[0])

plt.hist(paths_table['branch_distance'], bins=num_bins*50, range=possible_range, density=False, alpha=0.7)
plt.title('Branch Length Histogram')
plt.xlabel('Branch Length (µm)')
plt.ylabel('Number of Branches')
plt.grid(axis='y', alpha=0.75)
plt.show()

Using the histogram above and napari visualization, choose the minimum branch length you want to keep within your skeleton object

In [ ]:
### USER INPUT ###
min_branch_len = 0.3


### PROCESSING - no edits below ###
# select only branches that have end points
endpoint_indices = paths_table.index[(paths_table['deg_src'] == 1) | (paths_table['deg_dst'] == 1)]
print(f"Number of endpoint branches: {len(endpoint_indices)}")

# select branches that are the min size or below
short_paths_indices = paths_table.index[paths_table['branch_distance'] < min_branch_len]
print(f"Number of branches shorter than {min_branch_len} µm: {len(short_paths_indices)}")

# find indices that are endpoints and shorter than the determined size
indices_removed = pd.Index(set(endpoint_indices) & set(short_paths_indices))
print(f"Number of branches to be removed (endpoints and short): {len(indices_removed)}")

# remove those branches from the skeleton object & the paths table
pruned_skeleton = skel_g.prune_paths(indices_removed)
paths_table_pruned = paths_table.drop(indices_removed)

# visualize output
all_paths_pruned = [pruned_skeleton.path_coordinates(i) for i in range(pruned_skeleton.n_paths)]
viewer.add_shapes(all_paths_pruned, shape_type='path', properties=paths_table_pruned, edge_width=1, edge_color='skeleton_id', edge_colormap='hsv', scale=voxel_size_ZYX, opacity=1, name="Skeleton pruned")
paths_table_pruned.set_index(['skeleton_id', 'branch_id'], inplace=True)
paths_table_pruned.sort_index(inplace=True)
display(paths_table_pruned)

#  convert skeleton to image for visualization
skel_g_image = pruned_skeleton.skeleton_image
viewer.add_labels(skel_g_image.astype(int), scale=voxel_size_ZYX, name="Pruned skeleton labels", opacity=0.6)

##### ***Optional:* save the skeleton image for reference later**

In [ ]:
### USER INPUTS ###
skel_output_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair_5/3D_STED/WT_seg-skel_20260427-single-out" 

### PROCESSING - no edits below ###
# save skeleton image to output path
skimage.io.imsave(f"{skel_output_path}/{str(file_list[file_index].stem)}-PNN_skeleton.tif", skel_g_image, plugin="tifffile")

### **4. Measure PNN features**

#### **4A. Object size/shape and intensity measures per PNN objects and whole PNN**

Here, the instanace segmentation is quantified per PNN object and from the whole PNN (all PNN object combined into one)

In [ ]:
### PROCESSING - no edits below ###

### CONTNUING to quantification
# list properties to include in regionprops analysis
properties = ['label', 'bbox', 'centroid', 'num_pixels', 'area', 'equivalent_diameter', 
              'major_axis_length', 'minor_axis_length', 'extent', 'solidity', 'euler_number',
              'min_intensity', 'max_intensity', 'mean_intensity', 'intensity_std']

# process regionprops analysis for EACH PNN OBJECT SEPARATELY
props_obj = skimage.measure.regionprops_table(label_image=instance_seg, 
                                            intensity_image=test_img,
                                            properties=properties,
                                            spacing=voxel_size_ZYX)

props_obj_tab = pd.DataFrame(props_obj)
props_obj_tab.insert(0, 'object', 'PNN fragment')

surface_area_tab = pd.DataFrame(surface_area_from_props(instance_seg, props_obj, voxel_size_ZYX), columns=['surface_area'])
props_obj_tab.insert(13, 'surface_area', surface_area_tab['surface_area'])

# process regionprops analysis for WHOLE PNN OBJECT (one per image)
# ensure semantic segmentation only constists of 1 object (ID=1)
whole_PNN = (filled>0).astype(np.uint8)

props_whole = skimage.measure.regionprops_table(label_image=whole_PNN, 
                                                intensity_image=test_img,
                                                properties=properties,
                                                spacing=voxel_size_ZYX)
props_whole_tab = pd.DataFrame(props_whole)
props_whole_tab.insert(0, 'object', 'whole PNN')

surface_area_tab = pd.DataFrame(surface_area_from_props(whole_PNN, props_whole, voxel_size_ZYX), columns=['surface_area'])
props_whole_tab.insert(13, 'surface_area', surface_area_tab['surface_area'])


# combine both tables
combined_props_tab = pd.concat([props_obj_tab, props_whole_tab], ignore_index=True)

# rename & add columns for clarity and additional information
combined_props_tab.insert(0, 'image_name', file_list[file_index].name)
combined_props_tab.rename(columns={'area':'volume'}, inplace=True)
combined_props_tab['intensity_sum'] = combined_props_tab['mean_intensity'] * combined_props_tab['num_pixels']
combined_props_tab.insert(15, "SA_to_volume_ratio", combined_props_tab["surface_area"].div(combined_props_tab["volume"]))
rounded_scale = tuple(round(x, 2) for x in voxel_size_ZYX)
combined_props_tab.insert(1, "scale", str(rounded_scale))

display(combined_props_tab)

#### **~~4A. Intensity of WFA in PNN segmentation~~** <mark> ORIGINAL VERSION FOR WHOLE PNN ONLY - before using regionprops


In [ ]:
# ### PROCESSING - no edits below ###
# # select intensity values only where the PNN is present
# PNN_ints = test_img[filled > 0]

# # find unique PNN object IDs for counting
# unique = np.unique(instance_seg)
# unique = unique[unique != 0]

# # measure sum, mean, median, and standard deviation of intensity values
# int_dict = {"image": [str(file_list[file_index])],
#             "scale": [voxel_size_ZYX],
#             "PNN fragment count": [len(unique)],
#             "PNN total volume (voxels)": [np.count_nonzero(filled)],
#             "PNN total volume (um)": [np.count_nonzero(filled)*voxel_size_ZYX[0]*voxel_size_ZYX[1]*voxel_size_ZYX[2]],
#             "WFA total intensity (AU) in PNN": [np.sum(PNN_ints)],
#             "WFA mean intensity (AU) in PNN": [np.mean(PNN_ints)],
#             "WFA media intensity (AU) in PNN": [np.median(PNN_ints)],
#             "WFA SD intensity (AU) in PNN": [np.std(PNN_ints)]}

# PNN_int_tab = pd.DataFrame(int_dict)
# PNN_int_tab

#### **4B. Skeleton metrics**

The first table shows all the metrics that are collected using the skan package. Those metrics have been summarized per skeleton object and for all skeleton objects in the entire image.

In [ ]:
# ### PROCESSING - no edits below ###
# summarize skeleton branch table per skeleton object
paths_table_pruned.reset_index(inplace=True)
skel_sum1 = paths_table_pruned[['skeleton_id', 'branch_id']].groupby('skeleton_id').agg(['count'])
skel_sum2 = paths_table_pruned[['skeleton_id', 'branch_type']].groupby('skeleton_id').agg(['mean', 'median', 'min', 'max', 'std'])
skel_sum3 = paths_table_pruned[['skeleton_id', 'branch_distance', 'euclidean_distance']].groupby('skeleton_id').agg(['sum', 'mean', 'median', 'min', 'max', 'std'])

skel_summary = pd.concat([skel_sum1, skel_sum2, skel_sum3], axis=1)
skel_summary.columns = ['_'.join(col).strip() for col in skel_summary.columns.values]
skel_summary.reset_index(inplace=True)
skel_summary.insert(0, 'image_name', file_list[file_index].name)
skel_summary.insert(1, "scale", str(rounded_scale))
skel_summary.insert(2, 'object', 'PNN fragment')
skel_summary.rename(columns={'skeleton_id':'label',
                             'branch_id_count':'branch_count'}, inplace=True)

# summarize skeleton branch table for whole image
paths_table_pruned.insert(0, 'combined_skeleton_id', 1)  # assign all branches to skeleton ID 1 for whole image summary
combo_skel_sum1 = paths_table_pruned[['combined_skeleton_id', 'branch_id']].groupby('combined_skeleton_id').agg(['count'])
combo_skel_sum2 = paths_table_pruned[['combined_skeleton_id', 'branch_type']].groupby('combined_skeleton_id').agg(['mean', 'median', 'min', 'max', 'std'])
combo_skel_sum3 = paths_table_pruned[['combined_skeleton_id', 'branch_distance', 'euclidean_distance']].groupby('combined_skeleton_id').agg(['sum', 'mean', 'median', 'min', 'max', 'std'])

combo_skel_summary = pd.concat([combo_skel_sum1, combo_skel_sum2, combo_skel_sum3], axis=1)
combo_skel_summary.columns = ['_'.join(col).strip() for col in combo_skel_summary.columns.values]
combo_skel_summary.reset_index(inplace=True)
combo_skel_summary.insert(0, 'image_name', file_list[file_index].name)
combo_skel_summary.insert(1, "scale", str(rounded_scale))
combo_skel_summary.insert(2, 'object', 'whole PNN')
combo_skel_summary.rename(columns={'combined_skeleton_id':'label',
                                   'branch_id_count':'branch_count'}, inplace=True)

# combine both tables and format
final_skel_summary = pd.concat([skel_summary, combo_skel_summary], axis=0)

final_skel_summary

In [ ]:
### ORIGINAL CODE FROM BEFORE REGIONPROPS WAS INCLUDED - KEPT FOR REFERENCE ###

# ### PROCESSING - no edits below ###
# # skan table
# display(paths_table_pruned)

# # creating summary of skeleton metrics
# sekl_dict = {"image": [str(file_list[file_index])],
#             "scale": [voxel_size_ZYX],
#             "Total branch count": [paths_table.index.max()+1],
#             "Mean branch length (um)": [np.mean(paths_table["branch_distance"])],
#             "Median branch length (um)": [np.median(paths_table["branch_distance"])],
#             "SD branch length (um)": [np.std(paths_table["branch_distance"])]}

# skel_sum_tab = pd.DataFrame(sekl_dict)
# skel_sum_tab

#### **4C. Combine tables above for output**

In [ ]:
### PROCESSING - no edits below ###
combo = pd.merge(combined_props_tab, final_skel_summary, on= ['image_name', 'scale', 'object', 'label'], how='outer')
combo

##### ***Optional:* save the output table for reference later**

In [ ]:
### USER INPUTS ###
quant_output_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair_5/3D_STED/WT_seg-skel_20260427-single-out" 

### PROCESSING - no edits below ###
# save output table to output path
combo.to_csv(f"{quant_output_path}/{str(file_list[file_index].stem)}-PNN_quant.csv", index=False, mode='w')

----------

## **STEP 3. Batch Process segmentation and skeletonization (all images in one data folder):**

The above code is put together into a single function that enables you to batch process all of the images in a folder.

### **Segmentation & skeletonization**

First, we will segment and skeletonize all of the cells from each folder. After segmentation is complete, you will quality check the images for accuracy, and then the segmented images can be quantified and summarized (see below). 

There are two approaches you can use:
1. Batch process each file in a single folder **in parallel** as a separate job on the computing cluster (*RECOMMENDED, much faster*)
2. Batch process each file in a single folder **sequentially** using the function below or the [seg_skel_batch.py](/batch-scripts/seg_skel_batch.py) script (limited to 10 hrs when on an interactive node of the computing cluster)

#### **Parallel processing:**
Parallel processing allows you to simultaneously process each image in a directory. This cuts down the processing time SIGNIFICANTLY. 
1. Copy the [seg_skel_batch-parallel.py](/batch-scripts/seg_skel_batch-parallel.py) and [batch_process_seg_skel.sh](/batch-scripts/batch_process_seg_skel.sh) files into the desired output_path directory for the batch of data you wish to process. 
2. Edit the copied [seg_skel_batch-parallel.py](/batch-scripts/seg_skel_batch-parallel.py) to include the segmentation/skeletonization settings (excluding the file_path, file_type, and out_path) determined above and the path to the local copy of your PNN-morpho-quant repository.
3. Edit the copied [batch_process_seg_skel.sh](/batch-scripts/batch_process_seg_skel.sh) to include the input_dir (path to the directory containing the raw data), output_dir (path to the directory the segmentation and skeletonization files will be saved to), and the file_type (extension of the raw files). *These variables should NOT be specified in the [seg_skel_batch-parallel.py](/batch-scripts/seg_skel_batch-parallel.py) file since they are specified here.*
4. Initiate the batch script from the terminal with: 
    ```bash 
    sh batch_process_seg_skel.sh
    ```

*Important notes:*
- This process initiates a separate job on the compute cluster for each raw image in the input directory.
- The output_dir can be a new directory – it will be created as the first step in the script. 
- File paths should NOT have spaces.
- A log file is created for each image in the output folder. This tracks the information printed during the execution of the segmentation function and will include any error messages that occur.


#### **Sequential processing:**
If you wish to process images directly from this Jupyter notebook, you can use the code block below to do so. You could also initiate the same function using the [seg_skel_batch.py](/batch-scripts/seg_skel_batch.py) from the terminal:
    ```bash
    python seg_skel_batch.py
    ```

In either case, the raw files in the input directory will be processed one after the next making the processing time MUCH longer than the parallel processing approach above. If you are running this from an interactive node on the computing cluster, this process is limited to the 10 hr maximum processing time.


#### **Input variables:**
In either scenario, the following inputs need to be specified:
- `file_path`: Directory containing the raw input images to process.
- `file_type`: File extension for input images (for example, `".tif"`).
- `out_path`: Directory where segmentation and skeleton output files will be saved.
- `gaus_sigma`: Gaussian smoothing sigma; larger values apply stronger blur. If not using, specify 0.
- `med_size`: Median filter neighborhood size; larger values increase smoothing. If not using, specify 0.
- `manual_threshold_cutoff`: Intensity cutoff for manual/global thresholding (voxels `>=` cutoff are kept). If not using, specify None.
- `auto_threshold_method`: Global automated threshold algorithm (`'otsu'`, `'multiotsu'`, `'li'`, `'yen'`, `'isodata'`, `'triangle'`, `'minimum'`, `'mean'`). If not using, specify None.
- `auto_threshold_adjust`: Multiplier applied to the auto threshold value (`1` = none, `<1` more inclusive, `>1` more strict).
- `auto_multiotsu_middle_to`: For `'multiotsu'` only, whether the middle class is assigned to `'foreground'` or `'background'`. If not using, specify None.
- `local_threshold_method`: Local/adaptive threshold algorithm (`'otsu'`, `'mean'`, `'gaussian'`). If not using, specify None.
- `local_threshold_adjust`: Multiplier applied to local threshold values (same interpretation as global adjust). If not using, specify None.
- `local_threshold_size`: Local neighborhood size used for adaptive thresholding (typically an odd integer). If not using, specify None.
- `local_gaussian_sigma`: Sigma used when `local_threshold_method='gaussian'`. If not using, specify None.
- `obj_min_diameter`: Minimum object diameter used to remove small segmented objects.
- `obj_method`: Object filtering mode: `'slices'` (2D per z-slice) or `'3D'` (volumetric).
- `hole_min_diameter`: Maximum hole diameter to fill during cleanup.
- `hole_method`: Hole-filling mode: `'slices'` (2D per z-slice) or `'3D'` (volumetric).
- `min_branch_len`: Minimum skeleton branch length to keep; shorter endpoint branches are pruned.

In [ ]:
batch_PNN_seg_skel(file_path="/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair 5/3D STED/WT",
                    file_type=".tif",
                    out_path="/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair 5/3D STED/WT_seg-skel_20260325-nb-test",
                    gaus_sigma=2,
                    med_size=8,
                    manual_threshold_cutoff=0.045,
                    auto_threshold_method=None, #'multiotsu',
                    auto_threshold_adjust=None, #0.55,
                    auto_multiotsu_middle_to=None, #'background',
                    local_threshold_method=None, #'otsu',
                    local_threshold_adjust=None, #1,
                    local_threshold_size=None, #51, #71,
                    local_gaussian_sigma=None,
                    obj_min_diameter=10,
                    obj_method='3D',
                    hole_min_diameter=0,
                    hole_method='slices',
                    min_branch_len=0.3)

----------

## **STEP 4. Quality Check segmentations:**

### **Quality check output**

Before continue on to the quantification step, have a look at each of your images to confirm that the segmentation and skeletonization settings used created a succificient outcome. If necessary, refine your segmentation settings, or manually edit individual cells before quantification. Utilize the few lines of code below to open each raw image with the corresponding segmentation and skeletonization images one at a time.

If individual cells require editting, use step 2 above to edit the settings from that individual image. Along the way, can edit the intermediate outputs using the draw feature in Napari if refining the settings isn't accurate enough. 

***Important:** keep track of the changes you make to the segementations and skeletonizations. They may impact downstream metrics in adverse ways. To determine this, you can check if any edited cells significantly differ from unedited cells based on the quantitative outcomes.*

#### **1. List files in path**

Specify the following information:
- `raw_file_path`: file path where the input images are located written as a string
- `raw_file_type`: input file type as a string (e.g., ".tif")
- `seg_file_path`: file path to the segmentation and skeletonization files

Then run the cell below to read in the list of files of the specified type from the specified location. A Napari window will also pop up. The outputs of each processing step below will be added to the window as new layers.

In [8]:
### USER INPUTS ###
raw_file_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair_5/3D_STED/WT"  # OPTIONS:  WT" cKO"
raw_file_type = ".tif"
seg_file_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair_5/3D_STED/WT_seg-skel_20260427-sh-batch-test"  # OPTIONS:  WT" cKO"



### PROCESSING - no edits below ###
# open a Napari viewer window to visualize images & processing steps
viewer = napari.Viewer()

# create sorted list of files in directory with specified file type
file_list = sorted(Path(raw_file_path).glob(f"*{raw_file_type}"))

# print list of files with associated index number for selection below
pd.set_option('display.max_colwidth', None)
pd.DataFrame({"Image Name":file_list})

,Image Name
0,/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair_5/3D_STED/WT/2_VCX_382_7_decon_ch02.tif
1,/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair_5/3D_STED/WT/3_VCX_382_7_decon_ch02.tif
2,/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair_5/3D_STED/WT/4_VCX_382_7_decon_ch02.tif
3,/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair_5/3D_STED/WT/5_VCX_382_7_decon_ch02.tif
4,/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair_5/3D_STED/WT/6_VCX_382_7_decon_ch02.tif


#### **2. Select file of interest**

Specify the following information:
- `file_index`: the index of the image you would like to look at. The index value is found to the left of the file paths listed in the table above.

Then run the cell below to read in the image, associated processing files, and visualize them in Napari.

***Repeat this set for each image in your dataset.***

In [9]:
### USER INPUTS ###
file_index = 0  # change this number to select a different file from the list above



### PROCESSING - no edits below ###
# read in raw file
file_path = file_list[file_index]
raw_file = BioImage(str(file_path))
raw_img = np.squeeze(raw_file.data)

voxel_size_ZYX = (raw_file.physical_pixel_sizes.Z, raw_file.physical_pixel_sizes.Y, raw_file.physical_pixel_sizes.X)

# read in segmentation and skeletonization files
seg = skimage.io.imread(seg_file_path+ f"/{str(file_path.stem)}-PNN_instance_seg.tif")
skel = skimage.io.imread(seg_file_path+ f"/{str(file_path.stem)}-PNN_skeleton.tif")

# visualize image in napari
viewer.layers.clear()
viewer.add_image(raw_img, scale=voxel_size_ZYX, name="Deconvolved PNN image")
viewer.add_labels(seg, scale=voxel_size_ZYX, name="Segmentation", blending="additive", opacity=0.4)
viewer.add_labels(skel, scale=voxel_size_ZYX, name="Skeleton", blending="additive", opacity=0.4)

Attempted file (/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair_5/3D_STED/WT/2_VCX_382_7_decon_ch02.tif) load with reader: <class 'bioio_ome_tiff.reader.Reader'> failed with error: bioio-ome-tiff does not support the image: '/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair_5/3D_STED/WT/2_VCX_382_7_decon_ch02.tif'. Failed to parse XML for the provided file. Error: not well-formed (invalid token): line 1, column 6


<Labels layer 'Skeleton' at 0x7fb372a849b0>

#### **3. Repeat for all files in the dataset**

Note if your segmentation/skeletonization approach worked for this image. If so, continue to the next image by running Step 4-2 above with the next index value. If your segmentation and/or skeletonization approach did not work, return to Step 2 above to adjust the settings for this image (or the entire batch if necessary).

----------

## **STEP 5. Batch Process quantification (all images in one data folder):**

Now that the segmentation and skeletonization outputs have been quality checked, we will quantify the PNN morphology features from each image.

### **PNN Quantification**

First, we will quantify the PNN morphology per PNN object in each image from each folder/batch of data separately. After the morphology quantification is complete for all necessary datasets, they will be summarized per image (see below). 

There are two approaches you can use:
1. Batch process each file in a single folder **in parallel** as a separate job on the computing cluster (*RECOMMENDED, much faster*)
2. Batch process each file in a single folder **sequentially** using the function below or the [quant_batch.py](/batch-scripts/quant_batch.py) script(limited to 10 hrs when on an interactive node of the computing cluster)

#### **Parallel processing:**
Parallel processing allows you to simultaneously process each image in a directory. This cuts down the processing time SIGNIFICANTLY. 
1. Copy the [quant_batch-parallel.py](/batch-scripts/quant_batch-parallel.py) and [batch_process_quant.sh](/batch-scripts/batch_process_quant.sh) files into the desired quantification output directory for the batch of data you wish to process. 
2. Edit the copied [quant_batch-parallel.py](/batch-scripts/quant_batch-parallel.py) to specify if you wish to include the sureface area measurements in the quantification (this can increase the quantification time a lot) and the path to the local copy of your PNN-morpho-quant repository.
3. Edit the copied [batch_process_quant.sh](/batch-scripts/batch_process_quant.sh) to include the file_prefix (unique name prefix for the .csv output file), rawfile_dir (path to the directory that includes the raw image files), seg_skel_dir (path to the directory that includes the matching segmentation and skeletonization image files), quant_dri ((path to the directory that the output quantification .csv file will be saved),) *These variables should NOT be specified in the [quant_batch-parallel.py](/batch-scripts/quant_batch-parallel.py) file since they are specified here.*
4. Initiate the batch script from the terminal with: 
    ```bash 
    sh batch_process_quant.sh
    ```

*Important notes:*
- This process initiates a separate job on the compute cluster for each raw image in the raw file directory. The quantification results will be appended to the same CSV file.
- The quantification directory can be a new directory – it will be created as the first step in the script. 
- If a CSV file of the same name already exists in the specified quantification directory, the batch jobs will not be submitted and an error message will be shown in the terminal.
- File paths should NOT have spaces.
- A log file is created for each image in the output folder. This tracks the information printed during the execution of the segmentation function and will include any error messages that occur.


#### **Sequential processing:**
If you wish to process images directly from this Jupyter notebook, you can use the code block below to do so. You could also initiate the same function using the [quant_batch.py](/batch-scripts/quant_batch.py) from the terminal:
    ```bash
    python quant_batch.py
    ```

In either case, the files in the raw file directory will be processed one after the next making the processing time MUCH longer than the parallel processing approach above. If you are running this from an interactive node on the computing cluster, this process is limited to the 10 hr maximum processing time.


#### **Input variables:**
In either scenario, the following inputs need to be specified:
- `file_out_prefix`: Prefix added to the output quantification filename. A dash is automatically inserted before the rest of the filename. Use a unique value such as a date plus a short note, for example `"20260309_test"`.
- `raw_file_path`: Path to the directory containing the raw input image files.
- `raw_file_type`: File extension for the raw input images as a string, for example `".tif"`.
- `seg_skel_path`: Path to the directory containing the matching segmentation and skeleton files for the raw images.
- `quant_out_path`: Path to the directory where the quantification `.csv` output file will be saved. If this directory does not already exist, it will be created automatically.
- `include_surface_area`: `True` or `False` indicating whether to include surface area measurements in the quantification output. This is the most time-intensive part of the analysis.

In [ ]:
batch_PNN_quant(file_out_prefix="20260427-one-test",
                 raw_file_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair_5/3D_STED/WT-one-test", 
                 raw_file_type = ".tif",
                 seg_skel_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair_5/3D_STED/WT_seg-skel_20260427-one-test",
                 quant_out_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair_5/3D_STED/WT_quant_20260427-one-test",
                 include_surface_area = True)

----------

## **STEP 6. Summarize quantitative data per image (quantitative data from multiple folders):**

Once all of the biological replicates have undergone quantification, the quantitative data can be summarized per image (or in this case, per cell, since each image only contains the PNN from one cell).

This process should be very fast, so it can be run directly in the notebook without the need for parallelarization or batch scripting as done above.

Specify the following information and run the block below to summarize your data:
- `out_file_prefix`: prefix to append to the summary output file name when saving; a dash will be automatically added between the prefix and the rest of the file name. This allows for multiple unique rounds of summarization to be done and saved to the same location, if necessary. A good example of a prefix is the date of summarization and a brief note about the parameters used (e.g., "20260312_test")
- `csv_path_list`: A list of path for the .csv files to analyze. These should be the output quantification tables from the batch_PNN_quant function, and should all have the same column structure.
- `out_path`: A path string where the summary data file will be output to.

In [ ]:
batch_summarize_quant(out_file_prefix="20260427-one-test",
                       csv_path_list=["/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair_5/3D_STED/WT_quant_20260427-one-test/20260427-one-test-PNN_quantification.csv"],
                       out_path="/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair_5/3D_STED/WT_quant-summary_20260427-one-test")

All tables have surface area included. Surface area will be included in the summary table.
Summary table saved to: /users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair_5/3D_STED/WT_quant-summary_20260427-one-test/20260427-one-test-PNN_quant_summary.csv


---------------------------
## **Versioning Notes:**
- **V1.2**: 03/19/2026 SR includes batch script options for datasets that require more than 10 hours of processing time.
    - Batch processing of the segmentation/skeletonization and quantification are still included in the notebook
    - Instructions on how to batch process in parallel on the computing cluster are also now included in the markdown notes.
- **V1.1**: 12/11/2025 SR updates based on feedback from HSO
    - Load file (no change)
    - Segment PNN from WFA channel
        - Thresholding: update to add local and automated thresholding methods
    - Skeletonize PNN segmentation (no change)
    - Quantify PNN morphology & marker intensity:
        - Adding per PNN object and per PNN morphology quantification (skimage regionprops) and per object/PNN intensity quant of some additional channels (if they are the same resolution)

- **V1.0**: 10/28/2025 SR creates first draft with the following goals
    - Load file
        - list files of a specific type in path
        - read files (BioIO)
    - Segment PNN from WFA channel
        - Rescale intensities: min value = 0, max value = 1
        - Background subtraction: None
        - Denoising: None
        - Smoothing: gaussian = 2, median = 8
        - Thresholding: manual (low pass filter)
        - Clean-up: remove small objects (<10, 3D), fill small holes (none)
        - Instance segmentation: connectivity-based
    - Skeletonize PNN segmentation
        - Create skeleton object (skimage & skan)
        - Refine skeleton: remove small end-point branches (<1 um)
    - Quantify PNN morphology & marker intensity:
        - Intensity within entire PNN (segmented area, not per object)
        - Count and skeleton morphology metrics (from skan)